# Week 1

## Linear Algebra

### Task 1

(1) (-2, 1, 3) 與 (4, 6, 1)**
各分量比值：4 / (-2) = -2，6 / 1 = 6 → 比值不一致。
**→ 不平行**

**(2) (1, 2) 與 (-3, -6)**
各分量比值：-3 / 1 = -3，-6 / 2 = -3 → 全部都是 -3。
**→ 平行**（**v** = -3 · **u**）

**(3) (1, -2, 0, 1) 與 (3, 0, 2, -5)**
各分量比值：3 / 1 = 3，但第 2 分量 0 / (-2) = 0 ≠ 3（而且兩向量的 0 出現在不同位置）。
**→ 不平行**

**(4) (10, 0, 2, -4, -8) 與 (5, 0, 1, -2, -4)**
各分量比值：5 / 10 = ½，1 / 2 = ½，-2 / -4 = ½，-4 / -8 = ½，第 2 分量同為 0 也相符。
**→ 平行**（**v** = ½ · **u**，等價於 **u** = 2**v**）

---

**結論**

| 向量對 | 結果 |
|---|---|
| (-2, 1, 3) 與 (4, 6, 1) | 不平行 |
| (1, 2) 與 (-3, -6) | 平行（*k* = -3） |
| (1, -2, 0, 1) 與 (3, 0, 2, -5) | 不平行 |
| (10, 0, 2, -4, -8) 與 (5, 0, 1, -2, -4) | 平行（*k* = ½） |


### Task 2

### Task 3

### Task 4

## Python

### Task 1

In [4]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
WeHelp DL ── Week 1, Task 1
抓取 PChome 24h「ASUS 館」(store/DSAA31) 所有商品 ID，寫入 products.txt（一行一個）。
限制：只用 Python 標準函式庫，不使用第三方套件。

【為什麼不直接抓 https://24h.pchome.com.tw/store/DSAA31 的 HTML？】
  該頁是動態渲染，原始 HTML 沒有商品資料；商品是前端再打一支 JSON API 取回的。
  做法：直接打那支 API。以下是實測歸納出的規格（用 DevTools→Network→Fetch/XHR 也能對到同一支）。

【API 規格（實測）】
  端點： https://ecapi.pchome.com.tw/ecshop/prodapi/v2/store/<館別>/prod
    • host 必須是 ecapi（ecapi-cdn 會回空白）。
    • PChome 慣例：路徑後「直接用 & 接參數」，沒有問號 ?。
    • 參數 fields 與 _callback 都是必填，少一個就回空白。
    • 回應是 JSONP：  try{<callback>([...]);}catch(e){...}  ，要剝殼取出中間的 JSON 陣列。
    • offset 是「記錄序號」(從 1 起算)，limit 是每次期望筆數，但單次最多只回 36 筆。
      → 分頁必須用「offset += 實際回傳筆數」，不能用 offset += limit（否則會跳號漏抓）。
    • 超出範圍時回傳空陣列 []，即為結束訊號（不必預先知道總頁數）。
"""

import json
import time
import urllib.request
import urllib.error

# ─── 設定區 ──────────────────────────────────────────────
STORE_ID = "DSAA31"
API_URL = (
    "https://ecapi.pchome.com.tw/ecshop/prodapi/v2/store/"
    f"{STORE_ID}/prod"
    "&offset={offset}&limit={limit}&fields=Id,Name,Price&_callback=jsonp_cb"
)
LIMIT = 40            # 每次請求的期望筆數（API 實際上限約 36，靠動態步進自動吸收）
OFFSET_START = 1      # offset 從 1 起算（記錄序號）
MAX_PAGES = 200       # 安全上限，避免邏輯出錯時無限迴圈
SLEEP_SEC = 0.3       # 每次請求之間的禮貌延遲
OUTPUT_FILE = "products.txt"
HEADERS = {
    "User-Agent": ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                   "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36"),
    "Referer": f"https://24h.pchome.com.tw/store/{STORE_ID}",
}


def fetch(url):
    """送出 GET 請求，回傳回應文字。"""
    req = urllib.request.Request(url, headers=HEADERS)
    with urllib.request.urlopen(req, timeout=15) as resp:
        return resp.read().decode("utf-8", errors="replace")


def extract_json(text):
    """
    從 JSONP 取出 JSON 本體。回應形如：try{jsonp_cb([...]);}catch(e){...}
    作法：跳到 callback 第一個 '(' 之後，再用 raw_decode 解析「第一個 JSON 值」，
          它會自動忽略尾端的  );}catch...  包裹，不必手動找結尾。
          （若哪天 API 直接回純 JSON，以 { 或 [ 開頭，也照樣可解析。）
    """
    text = text.strip()
    if not text.startswith(("{", "[")):
        lp = text.find("(")
        if lp != -1:
            text = text[lp + 1:]
    obj, _ = json.JSONDecoder().raw_decode(text.lstrip())
    return obj


def parse_ids(data):
    """從一頁資料取出商品 ID 清單（本 API 回傳陣列；保留 dict 後備容錯）。"""
    if isinstance(data, list):
        rows = data
    elif isinstance(data, dict):
        rows = data.get("Rows") or next((v for v in data.values() if isinstance(v, list)), [])
    else:
        rows = []
    return [item["Id"] for item in rows if isinstance(item, dict) and "Id" in item]


def crawl_all():
    """逐頁抓取，直到某頁回傳空陣列為止（不預設總頁數）。"""
    seen = set()
    all_ids = []
    offset = OFFSET_START

    for page in range(1, MAX_PAGES + 1):
        url = API_URL.format(offset=offset, limit=LIMIT)
        try:
            data = extract_json(fetch(url))
        except urllib.error.HTTPError as e:
            print(f"[第 {page} 頁] HTTP 錯誤 {e.code}，停止。")
            break
        except urllib.error.URLError as e:
            print(f"[第 {page} 頁] 連線錯誤：{e.reason}，停止。")
            break

        ids = parse_ids(data)
        if not ids:                       # 空陣列 → 已到最後一頁
            print(f"[第 {page} 頁] 無資料(offset={offset})，爬取結束。")
            break

        for pid in ids:
            if pid not in seen:           # 去重保險
                seen.add(pid)
                all_ids.append(pid)
        print(f"[第 {page} 頁] offset={offset} 取回 {len(ids)} 筆，累計 {len(all_ids)} 筆。")

        offset += len(ids)                # ★ 關鍵：用實際筆數步進，避免漏抓
        time.sleep(SLEEP_SEC)

    return all_ids


def main():
    ids = crawl_all()
    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        for pid in ids:
            f.write(pid + "\n")
    # 依題目要求：一行一個印出商品 ID
    for pid in ids:
        print(pid)
    print(f"\n完成！共 {len(ids)} 個商品 ID，已寫入 {OUTPUT_FILE}")


if __name__ == "__main__":
    main()


[第 1 頁] offset=1 取回 36 筆，累計 36 筆。
[第 2 頁] offset=37 取回 10 筆，累計 46 筆。
[第 3 頁] 無資料(offset=47)，爬取結束。
DSAA31-A900JQR50-000
DSAA31-1900K0SXN-000
DSAA31-A900JQRLA-000
DSAA31-A900JTIUG-000
DSAA31-A900K19GS-000
DSAA31-1900JVJRC-000
DSAA31-A900K23IM-000
DSAA31-A900JQRNZ-000
DSAA31-A900JQR5L-000
DSAA31-1900JZ4LO-000
DSAA31-1900JZ44W-000
DSAA31-A900K1RRX-000
DSAA31-A900JQR53-000
DSAA31-A900K1VYC-000
DSAA31-1900K1BBW-000
DSAA31-A900IFZBC-000
DSAAOC-1900JZ40C-000
DSAA31-A900K282V-000
DSAA31-A900K1RRB-000
DSAA31-A900IFZEN-000
DSAA31-A900JQRNY-000
DSAA31-A900J29WS-000
DSAA31-A900J1950-000
DSAA31-A900IDS7J-000
DSAA31-A900JTLW5-000
DSAA31-1900JZ4TK-000
DSAA31-A900IV5FM-000
DSAA31-1900JVJNR-000
DSAA31-A900JQR4K-000
DSAA31-A900IVERT-000
DSAA31-1900JZZ83-000
DSAA31-1900K0X0E-000
DSAA31-A900JVF0M-000
DSAA31-A900JVF8Y-000
DSAA31-1900JZQ5I-000
DSAA31-1900K0XYQ-000
DSAA31-A900K1WE4-000
DSAA31-A900JQRLX-000
DSAA31-A900K23KB-000
DSAA31-A900K283O-000
DSAA31-A900K2SUT-000
DSAA31-A900K38PA-000
DSAA31-A900K3C69-000

### Task 2

In [5]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
WeHelp DL ── Week 1, Task 2
延續 Task 1 抓到的商品，篩出「至少 1 則評論、且平均評分 > 4.9」者，寫入 best-products.txt（一行一個）。
※ 請先執行上面 Task 1 的 cell —— 本格會重用它定義的 fetch / extract_json，並讀取它輸出的 products.txt。

【評分資料怎麼來（實測）】
  不需另外找評論服務，「同一支商品 API」指定 fields 就能拿到評論彙總：
    https://ecapi.pchome.com.tw/ecshop/prodapi/v2/prod/<商品ID>&fields=Id,ReviewCount,RatingValue&_callback=cb
    • ReviewCount = 評論則數
    • RatingValue = 平均評分（沒有評論時，這兩個欄位可能不出現）
  回應是 JSONP，本體是「以商品 ID 為鍵」的字典：
    {"<商品ID>": {"Id": "...", "ReviewCount": 1, "RatingValue": 5}}
  （多個 id 用逗號批次查實測無效，只會回第一筆，因此逐顆查詢。）
"""

import time

PROD_API = ("https://ecapi.pchome.com.tw/ecshop/prodapi/v2/prod/"
            "{pid}&fields=Id,ReviewCount,RatingValue&_callback=jsonp_cb")
SRC_FILE = "products.txt"          # Task 1 的輸出
OUTPUT_FILE = "best-products.txt"
MIN_REVIEWS = 1
MIN_RATING = 4.9                   # 注意是「嚴格大於」4.9（剛好 4.9 不算）
SLEEP_SEC = 0.2


def load_product_ids(path):
    """讀取 Task 1 產生的商品 ID（一行一個）。"""
    with open(path, "r", encoding="utf-8") as f:
        return [line.strip() for line in f if line.strip()]


def get_review_summary(pid):
    """查單一商品的評論彙總，回傳 (評論則數, 平均評分)；查不到則為 (0, None)。"""
    data = extract_json(fetch(PROD_API.format(pid=pid)))   # 重用 Task 1 的 fetch / extract_json
    info = data.get(pid, {}) if isinstance(data, dict) else {}
    return info.get("ReviewCount", 0) or 0, info.get("RatingValue")


def find_best_products(ids):
    """逐顆查詢並套用篩選條件。"""
    best = []
    for pid in ids:
        count, rating = get_review_summary(pid)
        if count >= MIN_REVIEWS and rating is not None and rating > MIN_RATING:
            best.append(pid)
            print(f"  ✓ {pid}  評論 {count} 則，平均 {rating}")
        time.sleep(SLEEP_SEC)
    return best


def main():
    ids = load_product_ids(SRC_FILE)
    print(f"讀入 {len(ids)} 個商品，開始查詢評論彙總…")
    best = find_best_products(ids)

    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        for pid in best:
            f.write(pid + "\n")
    # 依題目要求：一行一個印出符合條件的商品 ID
    print(f"\n=== 至少 {MIN_REVIEWS} 則評論且平均評分 > {MIN_RATING} 的商品（共 {len(best)} 筆）===")
    for pid in best:
        print(pid)
    print(f"\n完成！已寫入 {OUTPUT_FILE}")


if __name__ == "__main__":
    main()


讀入 46 個商品，開始查詢評論彙總…
  ✓ DSAA31-A900JQR50-000  評論 1 則，平均 5
  ✓ DSAA31-1900JZ4LO-000  評論 4 則，平均 5
  ✓ DSAA31-A900IFZBC-000  評論 1 則，平均 5
  ✓ DSAA31-A900IFZEN-000  評論 1 則，平均 5
  ✓ DSAA31-A900J1950-000  評論 1 則，平均 5

=== 至少 1 則評論且平均評分 > 4.9 的商品（共 5 筆）===
DSAA31-A900JQR50-000
DSAA31-1900JZ4LO-000
DSAA31-A900IFZBC-000
DSAA31-A900IFZEN-000
DSAA31-A900J1950-000

完成！已寫入 best-products.txt


### Task 3

In [6]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
WeHelp DL ── Week 1, Task 3
延續 Task 1 抓到的商品，算出「Intel i5 處理器」ASUS PC 的平均價格，直接印在 console。
※ 請先執行 Task 1 的 cell —— 本格會重用它定義的 fetch / extract_json，並讀取它輸出的 products.txt。

【怎麼判斷 i5、價格從哪來（實測）】
  同一支商品 API 指定 fields 即可拿到名稱與價格：
    https://ecapi.pchome.com.tw/ecshop/prodapi/v2/prod/<商品ID>&fields=Id,Name,Price&_callback=cb
  • 處理器寫在商品名稱裡，格式像 "(i5-14500/32G/1TB SSD/W11)"。
    → 判斷「Intel i5」= 名稱（不分大小寫）含 "i5"。
    ※ 注意品牌差異：C5-210H 是 Core 5、U5-225 是 Core Ultra 5、R5-220 非 Intel i5，皆不計入。
  • 價格取 Price.P（實際售價）。
"""

import re
import time

PROD_API = ("https://ecapi.pchome.com.tw/ecshop/prodapi/v2/prod/"
            "{pid}&fields=Id,Name,Price&_callback=jsonp_cb")
SRC_FILE = "products.txt"          # Task 1 的輸出
SLEEP_SEC = 0.2


def load_product_ids(path):
    """讀取 Task 1 產生的商品 ID（一行一個）。"""
    with open(path, "r", encoding="utf-8") as f:
        return [line.strip() for line in f if line.strip()]


def get_name_price(pid):
    """查單一商品，回傳 (名稱, 售價)；售價取不到則為 None。"""
    data = extract_json(fetch(PROD_API.format(pid=pid)))   # 重用 Task 1 的 fetch / extract_json
    info = data.get(pid, {}) if isinstance(data, dict) else {}
    price = (info.get("Price") or {}).get("P")
    return info.get("Name", ""), price


def is_intel_i5(name):
    """名稱含 i5（不分大小寫）視為 Intel i5；Core 5 / Core Ultra 5 不算。"""
    return re.search(r"i5", name, re.IGNORECASE) is not None


def main():
    ids = load_product_ids(SRC_FILE)
    prices = []
    for pid in ids:
        name, price = get_name_price(pid)
        if price and is_intel_i5(name):
            prices.append(price)
            print(f"  i5 ▸ {price:>6}　{name}")
        time.sleep(SLEEP_SEC)

    if prices:
        avg = sum(prices) / len(prices)
        print(f"\nIntel i5 ASUS PC 共 {len(prices)} 筆，總價 {sum(prices)}")
        print(f"平均價格 = {avg:.1f} 元")
    else:
        print("找不到含 Intel i5 處理器的商品。")


if __name__ == "__main__":
    main()


  i5 ▸  33990　華碩 H-S503MER-514500076W(i5-14500/32G/1TB SSD/W11)
  i5 ▸  29888　華碩 H-S500TER-514500005W(i5-14500/16G/1TB SSD/W11)
  i5 ▸  24888　華碩 H-V500MV-13420H162W(i5-13420H/16G/1TB SSD/W11)
  i5 ▸  29888　華碩 H-S500TER-514500005W(i5-14500/16G/1TB SSD/W11)

Intel i5 ASUS PC 共 4 筆，總價 118654
平均價格 = 29663.5 元


### Task 4

In [7]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
WeHelp DL ── Week 1, Task 4
延續 Task 1 抓到的商品，用 z-score 標準化價格，輸出到 standardization.csv 並印在 console。
※ 請先執行 Task 1 的 cell —— 本格會重用它定義的 fetch / extract_json，並讀取它輸出的 products.txt。

【z-score（標準分數）】
  z = (x - μ) / σ
  題目明講「把 Task 1 的資料當作母體(population)」，所以 σ 用「母體標準差」：
      σ = sqrt( Σ(x - μ)² / N )      ← 除以 N，不是 N-1（那是樣本標準差）
  z 的意義：這筆價格距離平均幾個標準差（正=偏貴、負=偏便宜、0=等於平均）。

【輸出格式】
  每列為「商品ID,價格,價格z分數」，不含表頭（題目範例兩行皆為此格式，是逐列樣板而非標題）：
      DSAA31-A900JQR50-000,22990,-1.077039
"""

import math
import time

PROD_API = ("https://ecapi.pchome.com.tw/ecshop/prodapi/v2/prod/"
            "{pid}&fields=Id,Price&_callback=jsonp_cb")
SRC_FILE = "products.txt"          # Task 1 的輸出
OUTPUT_FILE = "standardization.csv"
SLEEP_SEC = 0.2
ZSCORE_NDIGITS = 6                 # z 分數小數位數（純粹為了輸出整齊）


def load_product_ids(path):
    """讀取 Task 1 產生的商品 ID（一行一個）。"""
    with open(path, "r", encoding="utf-8") as f:
        return [line.strip() for line in f if line.strip()]


def get_price(pid):
    """查單一商品售價 Price.P；取不到回 None。"""
    data = extract_json(fetch(PROD_API.format(pid=pid)))   # 重用 Task 1 的 fetch / extract_json
    info = data.get(pid, {}) if isinstance(data, dict) else {}
    return (info.get("Price") or {}).get("P")


def population_mean_std(values):
    """回傳 (平均, 母體標準差)；母體標準差除以 N。"""
    n = len(values)
    mean = sum(values) / n
    variance = sum((x - mean) ** 2 for x in values) / n
    return mean, math.sqrt(variance)


def main():
    ids = load_product_ids(SRC_FILE)

    # 先把所有價格抓回來（保留 Task 1 的順序）；算 z 分數需要先有整個母體的平均與標準差
    rows = []
    for pid in ids:
        price = get_price(pid)
        if price is not None:
            rows.append((pid, price))
        time.sleep(SLEEP_SEC)

    prices = [p for _, p in rows]
    mean, std = population_mean_std(prices)
    print(f"母體 N={len(prices)}，平均={mean:.4f}，母體標準差={std:.4f}\n")

    with open(OUTPUT_FILE, "w", encoding="utf-8", newline="") as f:
        for pid, price in rows:
            z = (price - mean) / std
            line = f"{pid},{price},{round(z, ZSCORE_NDIGITS)}"
            f.write(line + "\n")
            print(line)                # 依題目要求同時印在 console

    print(f"\n完成！共 {len(rows)} 筆，已寫入 {OUTPUT_FILE}")


if __name__ == "__main__":
    main()


母體 N=46，平均=32944.3913，母體標準差=9242.3672

DSAA31-A900JQR50-000,22990,-1.077039
DSAA31-1900K0SXN-000,35990,0.329527
DSAA31-A900JQRLA-000,18990,-1.509829
DSAA31-A900JTIUG-000,22990,-1.077039
DSAA31-A900K19GS-000,22990,-1.077039
DSAA31-1900JVJRC-000,23990,-0.968842
DSAA31-A900K23IM-000,23990,-0.968842
DSAA31-A900JQRNZ-000,30990,-0.21146
DSAA31-A900JQR5L-000,24990,-0.860644
DSAA31-1900JZ4LO-000,33990,0.113132
DSAA31-1900JZ44W-000,29990,-0.319657
DSAA31-A900K1RRX-000,25990,-0.752447
DSAA31-A900JQR53-000,21990,-1.185237
DSAA31-A900K1VYC-000,29888,-0.330694
DSAA31-1900K1BBW-000,24888,-0.871681
DSAA31-A900IFZBC-000,25888,-0.763483
DSAAOC-1900JZ40C-000,24990,-0.860644
DSAA31-A900K282V-000,32888,-0.006101
DSAA31-A900K1RRB-000,25990,-0.752447
DSAA31-A900IFZEN-000,47990,1.627896
DSAA31-A900JQRNY-000,47990,1.627896
DSAA31-A900J29WS-000,43990,1.195106
DSAA31-A900J1950-000,43990,1.195106
DSAA31-A900IDS7J-000,43990,1.195106
DSAA31-A900JTLW5-000,43990,1.195106
DSAA31-1900JZ4TK-000,47990,1.627896
DSAA31-A9